# ZTE — ZuCo Thought Embedding

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/victor-iyi/zte/blob/feature/decoder/notebooks/zte_colab_v2.ipynb)

**Reading EEG back into language, honestly.** This notebook is the operating manual for the ZTE pipeline: an
**encoder** that maps word- and sentence-level EEG into a shared embedding space, and a **decoder** that reads text
out of it through a frozen language model on a small trainable leash.

The north star is clinical — helping people with ALS, paralysis and locked-in syndrome turn EEG into natural
language. Work aimed at a vulnerable population has to earn trust, so **an overclaimed result is worse than no
result**. Everything below is built around that: bootstrap confidence intervals over point estimates, permutation
nulls, held-out splits that hold out whole *people*, provenance travelling with every artifact, and null results
reported plainly.

---

## What this notebook is for

| You want to | Go to |
| --- | --- |
| Set up a fresh Colab runtime | §1 – §3 |
| Mount Drive and prepare the data once | §4 – §5 |
| Understand what the current encoder can and cannot do | §6 |
| Train the encoder (one arm, the ablations, or the full sweep) | §7 |
| Train and audit the decoder | §8 |
| Run the entire study with one resumable command | §9 |
| Look at everything you have ever run, visually | §10 |
| Persist, resume and continue offline | §11 – §12 |

## How to read a number in this project

Four rules, each of which has already caught a wrong conclusion here.

1. **`held_out_retrieval` is the result. `sentence_retrieval` is not.** The pooled number is computed over the
   training subjects as well as the held-out one, so it rewards memorising the brains you have rather than reaching
   the one you do not. It inverted the champion once already.
2. **Top-1 on 700 queries expects exactly one hit by chance.** A "0.006 vs 0.001" headline is three hits at
   $p \approx 0.08$. Read **rank percentile** with its confidence interval, and read Top-K as *hit counts* with an
   exact binomial tail.
3. **Sentence length is 5.14 of the 9.45 bits.** ZuCo segments words by eye tracking, so the model gets the word
   count for free, and a length-only oracle beats every encoder measured here on every top-k. Any retrieval number
   quoted without saying whether it is length-stratified is not a claim.
4. **Generation is not a headline unless the verdict says so.** `verdict['generation_above_controls']` ANDs over an
   honest split, no candidate set, every pre-registered control beaten, a permutation $p < 0.05$, and a
   prefix-influence KL above the floor. A control that did not run *fails* its clause.

> **A synthetic run is never a result.** `--synthetic` exists to prove the plumbing works. Every number that is
> allowed to leave this notebook came from real ZuCo.

## 1 · Provision the runtime

Colab ships an older Python than ZTE requires (`>=3.14`), so `uv` provisions the pinned interpreter and installs
everything into a cached virtualenv. The clone is shallow and hard-resets to `main`, so only git-tracked files move —
your Drive folder and any local cache are untouched.

Re-running this cell on a warm runtime is fast and idempotent.

In [ ]:
%%bash
pip install -q uv
# Work whether this is a fresh runtime (/content), a re-run already inside zte/, or a restored session.
if [ -f pyproject.toml ]; then :
elif [ -d zte/.git ]; then cd zte
else git clone --depth 1 https://github.com/victor-iyi/zte.git --branch feature/decoder && cd zte
fi
git fetch --depth 1 origin feature/decoder && git reset --hard FETCH_HEAD
echo "ZTE @ $(git rev-parse --short HEAD): $(git log -1 --pretty=%s)"
uv python install 3.14
uv sync --all-groups

## 2 · Wire the kernel

Four environment variables that every later `!uv run` subprocess inherits. Each one fixes a failure that is silent
rather than loud, which is why they are set here and not left to chance.

In [ ]:
import os

# Enter the repo in the notebook kernel, so relative paths and every subprocess resolve. A %%bash `cd` cannot
# do this: it dies with its own shell.
if os.path.isdir('zte') and not os.path.isfile('pyproject.toml'):
    os.chdir('zte')

# Colab's inline matplotlib backend crashes a headless subprocess, so Agg is forced rather than defaulted.
os.environ['MPLBACKEND'] = 'Agg'
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

# Python block-buffers stdout when it is not a terminal, and a `!uv run` subprocess never is. Without this a
# multi-hour run looks silent and then dumps every line at once.
os.environ['PYTHONUNBUFFERED'] = '1'

# Let the CUDA allocator grow segments instead of fragmenting: a raw-EEG batch allocates a few very large
# activation blocks, which is exactly the pattern that strands free memory.
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('MPLCONFIGDIR', os.path.abspath('res/.cache/matplotlib'))

try:
    from google.colab import userdata  # type: ignore[import-untyped]

    _hf = userdata.get('HF_TOKEN')
except Exception as exc:  # not on Colab, or the secret is not granted to this notebook
    _hf, _ = None, print(f'HF_TOKEN unavailable ({type(exc).__name__}) — Hub downloads will be unauthenticated.')
if _hf:
    os.environ['HF_TOKEN'] = _hf
    print('HF_TOKEN loaded — authenticated HuggingFace Hub downloads enabled.')

!uv run python -c "from zte.utils import bootstrap; import json; print(json.dumps(bootstrap(chdir=True, quiet=True), default=str, indent=1))"

## 3 · What hardware did you get, and what will ZTE do with it

ZTE resolves one device spec and every component reads it — precision, DataLoader workers, pinned memory, static
shapes on TPU. The cell prints both what Colab gave you and what ZTE will do with it, so an out-of-memory kill later
is predictable rather than a mystery.

**A raw-conformer batch turns every (sentence, word) pair into its own 105×350 attention problem.** That is tens of
gigabytes of activations on a full batch, which is why `model.grad_checkpoint: true` is set in every raw config —
numerically identical gradients, roughly 30% slower, and the difference between fitting on a 16 GB GPU and not.

In [ ]:
%%bash
uv run python - <<'PY'
import json

from zte.device import auto_num_workers, resolve_device
from zte.utils.env import accelerator_info

spec = resolve_device('auto')
plan = {
    'backend': spec.kind,
    'device': spec.name,
    'autocast_dtype': str(spec.autocast_dtype).replace('torch.', '') if spec.autocast_dtype else 'fp32',
    'mixed_precision': spec.use_amp,
    'pin_memory': spec.supports_pin_memory,
    'dataloader_workers_auto': auto_num_workers(spec, -1),
    'tf32_matmul': spec.kind == 'cuda',
    'static_shapes': spec.kind == 'xla',
}
print(json.dumps({'detected': accelerator_info(), 'zte_will_use': plan}, indent=2))
PY

## 4 · Drive is the workspace

**Everything durable lives on Drive.** A Colab VM can vanish without warning; a multi-hour sweep whose only copy was
on the VM disk is a multi-hour sweep you get to run again.

The layout is one shared folder for data plus one dated folder per session:

```text
Sharables/ZTE/
├── prepared/                     # cached feature bundles, NOT date-stamped: built once, reused forever
├── ZuCo Dataset/                 # the raw .mat files
└── YYYY-MM-DD/                   # one folder per session
    ├── experiments/              # every run: config, checkpoints, evaluation, figures
    ├── analysis/                 # the study dashboard and its tidy tables
    └── archives/                 # provenance-stamped zips
```

**Where each kind of work writes.** Training checkpoints go to the VM's fast local disk and are mirrored to Drive
after every stage, because a Drive FUSE stall mid-`torch.save` is a torn checkpoint. Everything else — evaluation,
generation, the analysis dashboard, the archives — is written straight to Drive. Set `WRITE_MODE = 'drive'` below to
put checkpoints on Drive too; it is slower and less robust, and it survives a VM reset without the mirror step.

**To resume an interrupted session**, set `RESUME_DATE` to that session's folder name. Everything then points at the
same place and every `--resume` finds its work already done.

### What is safe at every moment

The rule is that nothing expensive is ever more than one epoch, or one stage, away from durable storage.

| written | when | why then |
| --- | --- | --- |
| `best.pt` | the moment it improves | it is the result; losing it loses the run's whole point |
| `last.pt` | every epoch | it is what `--resume` reads, so a reclaimed VM costs one epoch |
| `ckpt_epoch*.pt` | never mirrored | rotation history: `keep_last` extra copies of a large file that a fresh VM cannot use |
| the run directory | each stage | config, `history.json`, evaluation, figures, TensorBoard |
| evaluation · generation · analysis · studio | straight to the durable root | expensive, and none of it resumes -- recomputing is the only recovery |
| the prepared feature bundle | once, ever | content-addressed and *not* date-stamped, so every future session reuses it |

A mirror that fails must never kill a run, so failures are logged and training continues -- but a mirror that has
been silently failing for forty epochs is worse than one that failed loudly, so consecutive failures escalate to an
error naming the missing file.

Restoring a run directory from Drive gives you `best.pt` and `last.pt` and no rotation history. That is deliberate,
and `--resume` handles it: it tries `last.pt`, then any epoch files, then `best.pt`. Even a `last.pt` torn by the
write that was in flight when the machine went away costs the epochs since the last improvement, not the run.

In [ ]:
from google.colab import drive  # type: ignore[import-untyped]

drive.mount('/gdrive')

In [ ]:
import datetime
import glob
import json
import os
import pathlib
import shutil
import subprocess

# --- The one shared folder that holds everything -------------------------------------------------- #
ZTE_DRIVE: str = '/gdrive/My Drive/Sharables/ZTE'
DATA_DIR: str = f'{ZTE_DRIVE}/ZuCo Dataset'

# Set to an existing folder name (e.g. '2026-08-13') to resume that session; None starts today's.
RESUME_DATE: str | None = None
# 'local+mirror' trains on the VM disk and copies to Drive after each stage (recommended).
# 'drive' writes runs straight to Drive: slower, but nothing to mirror if the VM dies mid-epoch.
WRITE_MODE: str = 'local+mirror'

RUN_DATE: str = RESUME_DATE or datetime.date.today().isoformat()
DRIVE_DIR: str = f'{ZTE_DRIVE}/{RUN_DATE}'
DRIVE_RUNS: str = f'{DRIVE_DIR}/experiments'
DRIVE_ANALYSIS: str = f'{DRIVE_DIR}/analysis'
LOCAL_RUNS: str = 'res/experiments'
for path in (DRIVE_RUNS, DRIVE_ANALYSIS, f'{DRIVE_DIR}/archives'):
    os.makedirs(path, exist_ok=True)

# Where long runs write, and where they are always backed up to.
OUT_ROOT: str = DRIVE_RUNS if WRITE_MODE == 'drive' else LOCAL_RUNS
DRIVE_BACKUP: str = DRIVE_RUNS
os.environ.update(
    ZTE_DRIVE=ZTE_DRIVE,
    DATA_DIR=DATA_DIR,
    DRIVE_DIR=DRIVE_DIR,
    RUN_DATE=RUN_DATE,
    OUT_ROOT=OUT_ROOT,
    DRIVE_BACKUP=DRIVE_BACKUP,
    DRIVE_ANALYSIS=DRIVE_ANALYSIS,
)
print(f'session   : {RUN_DATE}   ({"resumed" if RESUME_DATE else "new"})')
print(f'raw data  : {DATA_DIR}   (present: {os.path.isdir(DATA_DIR)})')
print(f'runs ->   : {OUT_ROOT}   (mirrored to {DRIVE_BACKUP})')
print(f'analysis  : {DRIVE_ANALYSIS}')

### 4a · Helpers this notebook uses everywhere

Small, boring and worth reading once: where runs can be found, how much room you have, and how to move state
between the VM and Drive in both directions.

In [ ]:
def run_dirs(*extra: str) -> list[pathlib.Path]:
    """Every run directory reachable right now — this session's Drive folder first, then any extra root, then local.

    Runs land in different places depending on the cell that produced them, and a fresh runtime has an empty local
    disk, so the search order matters and duplicates are resolved by first-seen.
    """
    seen: set[str] = set()
    found: list[pathlib.Path] = []
    for base in (DRIVE_RUNS, *extra, LOCAL_RUNS):
        for manifest in sorted(glob.glob(f'{base}/*/manifest.json')):
            directory = pathlib.Path(manifest).parent
            if directory.name not in seen:
                seen.add(directory.name)
                found.append(directory)
    return found


def every_session() -> list[str]:
    """Every dated session folder on Drive, newest first — what the analysis section reads across."""
    dates = [p for p in sorted(glob.glob(f'{ZTE_DRIVE}/20*'), reverse=True) if os.path.isdir(f'{p}/experiments')]
    return [f'{p}/experiments' for p in dates]


def resolve_ckpt(run_name: str, which: str = 'best') -> str:
    """Finds a run's checkpoint, Drive first, so a fresh VM can evaluate a previous session without restoring it.

    Search order is this session's Drive folder, then every earlier session newest-first, then the local disk. On a
    reclaimed runtime the local disk is empty and Drive holds `best.pt` and `last.pt` for every run ever trained
    here, so evaluation, decoding and the studio all keep working with no manual step.
    """
    candidates = [
        f'{DRIVE_RUNS}/{run_name}/checkpoints/{which}.pt',
        *[f'{root}/{run_name}/checkpoints/{which}.pt' for root in every_session()],
        f'{LOCAL_RUNS}/{run_name}/checkpoints/{which}.pt',
    ]
    for path in candidates:
        if os.path.isfile(path):
            where = 'Drive' if path.startswith(ZTE_DRIVE) else 'local disk'
            print(f'{which}.pt for {run_name}: {where}\n  {path}')
            return path

    raise FileNotFoundError(f'no {which}.pt for {run_name!r} on Drive or locally; train it first (Section 7).')


def durable(*parts: str) -> str:
    """Returns a path under the durable root for this machine: the Drive session on Colab, `res/` locally.

    Everything expensive that does *not* need resuming -- the analysis dashboard, generation.jsonl, the studio page,
    the rebaseline audit -- is written here rather than written locally and copied later, so a VM reclaimed halfway
    through the *next* cell cannot take it.
    """
    root = DRIVE_DIR if os.path.isdir(ZTE_DRIVE) else 'res'
    path = os.path.join(root, *parts)
    os.makedirs(os.path.dirname(path) or path, exist_ok=True)
    return path


def show_resources() -> None:
    """Prints RAM / GPU / disk, so an out-of-memory kill is predictable rather than a mystery."""
    gb = 1 << 30
    total = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / gb
    print(f'System RAM: {total:5.1f} GB   |   free disk: {shutil.disk_usage(".").free / gb:5.1f} GB')
    try:
        import torch

        if torch.cuda.is_available():
            props = torch.cuda.get_device_properties(0)
            print(f'GPU:        {props.name} ({props.total_memory / gb:.1f} GB)')
        else:
            print('GPU:        none — Runtime → Change runtime type → GPU')
    except ImportError:
        print('GPU:        torch not importable in the notebook kernel (the uv env has it)')
    if total < 20:
        print('\n⚠  Raw-EEG bundles are ~24 GB materialised. Prefer Runtime → Change runtime type → High-RAM.')


_HEAVY = shutil.ignore_patterns('cache', 'tb', 'bundle', 'ckpt_epoch*.pt', 'last.pt')


def mirror_to_drive(local: str = LOCAL_RUNS, drive_sub: str = 'experiments') -> None:
    """Copy a local directory to Drive, minus the heavy transient files, so the session survives the VM."""
    src = pathlib.Path(local)
    if not src.is_dir():
        print('nothing to mirror at', local)
        return

    dst = pathlib.Path(DRIVE_DIR) / drive_sub
    shutil.copytree(src, dst, dirs_exist_ok=True, ignore=_HEAVY)
    print(f'mirrored {src} -> {dst}')


def restore_from_drive(run_date: str | None = None, drive_sub: str = 'experiments', local: str = LOCAL_RUNS) -> None:
    """Pull a session's runs back to the VM so `--resume` can continue them after a runtime reset."""
    src = pathlib.Path(f'{ZTE_DRIVE}/{run_date or RUN_DATE}/{drive_sub}')
    if not src.is_dir():
        print('nothing to restore at', src)
        return

    pathlib.Path(local).mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, pathlib.Path(local), dirs_exist_ok=True)
    print(f'restored {len(list(src.iterdir()))} item(s) from {src} -> {local}. Re-run any cell with --resume.')


show_resources()

## 5 · Prepare the data once, on Drive, and never again

`zte-prepare` keys every shipped config by a hash of the fields that actually change the processed bundle, asks the
persistent Drive store what it already holds, and builds only what is genuinely absent. A fully-prepared project
never touches the raw `.mat` files again.

Two paths, and the split matters:

- **`ZTE_CACHE_REMOTE`** → `Sharables/ZTE/prepared`. Persistent and *not* date-stamped, because a feature bundle is a
  property of the data and the config, not of the session that happened to build it.
- **`DATA_CACHE`** → a local copy on the VM's fast disk. Bundles are content-addressed and immutable, so staging is a
  copy-if-absent and never a re-computation.

Building the raw bundles from scratch is the single longest step in the project. Once it is on Drive it is done
forever, for every future session.

In [ ]:
PREPARED_DRIVE: str = f'{ZTE_DRIVE}/prepared'
PREPARED_LOCAL: str = 'res/cache/prepared'
os.environ['ZTE_CACHE_REMOTE'] = PREPARED_DRIVE
os.environ['DATA_CACHE'] = PREPARED_LOCAL

!uv run zte-prepare --root "{DATA_DIR}" --configs --cache-dir "{PREPARED_LOCAL}" --cache-remote "{PREPARED_DRIVE}"

### 5a · Is the signal even there? — the model-free audit

Before any model, `zte-audit` measures the associations in the data itself. Run it once per dataset and read it
before believing any result. Two findings from it govern every design decision below:

- **Task is an alias for the stimulus.** Cramér's $V(\text{task}, \text{stimulus}) = 0.998$ and *no* sentence appears
  under both tasks, so a task-invariance loss also deletes content. The answer is to match negatives within task, not
  to turn the adversary up.
- **Word length and word frequency are the same variable here** ($|\rho| = 0.99$), and eye-tracking measures carry
  subject identity ($\eta \approx 0.25$).

In [ ]:
!uv run zte-audit --root "{DATA_DIR}" --out "{DRIVE_ANALYSIS}"

## 6 · The encoder — where it stands, and what exp16 changes

### What has actually been measured

Real ZuCo, LOSO with `ZAB` held out, 160,804 words. These are the numbers the new work is designed against, not a
summary of them:

| measurement | value | reading |
| --- | --- | --- |
| Held-out rank percentile | **0.9636** [0.9599, 0.9674] | real, and the honest headline |
| Held-out Top-1 | 9 hits / 700, $p = 1.1 \times 10^{-6}$ | real, but nine events |
| Length-stratified rank percentile | **0.9211** [0.9154, 0.9270] | still real with length held constant |
| Variance budget | 8.4% subject · 0.0% content · 91.6% neither | nine tenths is single-trial noise |
| Same word, different subject | cosine gap **+0.005** | *not clustered* |
| Held-out `word_len` probe | $R^2 = -0.060$ | below chance |
| Decoder rescoring, length-stratified | rank percentile **0.4349** | below chance — it rides length alone |

And the sweep of 2026-07-25: thirteen arms flipping Euclidean alignment, the subject adapter and identity
orthogonality all landed between **2 and 9 hits in 700**. Run-to-run noise was the size of every effect. The exposed
levers are exhausted.

### The bit budget

Naming one of 700 sentences costs $\log_2 700 = 9.4512$ bits. Conditioning on word count leaves 4.3090, so length
alone supplies

$$
I(\text{identity};\, n_\text{words}) \;=\; H(\text{identity}) - H(\text{identity} \mid n_\text{words}) \;=\; 5.1422 \text{ bits}
$$

for free, and the encoder contributes roughly 1.5 on top. Free generation of a 19.6-word sentence needs about 190
bits — some **2.5%** of what is required. Expect an honest null on generation; the powered readout is retrieval.

### The four mechanisms

Each attacks one of the failures in the table, each is off by default, and each has a matched ablation that flips
exactly one lever.

| mechanism | config | the failure it attacks |
| --- | --- | --- |
| **Predictive residual coding** | `model.residual_coding` | 91.6% of variance is neither identity nor content |
| **Cross-reader consensus** | `objective.consensus_*` | same word, different subject: gap +0.005 |
| **Length-matched gallery contrast** | `objective.gallery_*` | 5.14 free bits; rescoring below chance once stratified |
| **Length projection** | `objective.length_projection` | a length oracle beats the encoder on every top-k |

**Predictive residual coding.** Reading is predictive, and the largest language-related EEG deflections are
*surprisal* responses. Everything unsurprising about a moment of reading — tonic state, cap impedance, the 1/f
background, the drift of the last few seconds — is predictable from the preceding words and cancels in the residual;
the word-specific response is not and does not. A causal head predicts each token from its left context and the
token keeps only what was left over. The head trains on its own regression against a *detached* target, so the
encoder cannot cut that loss by making itself predictable, which would be collapse wearing a disguise.

**Cross-reader consensus.** ZuCo gives all twelve subjects the same 700 sentences, so each stimulus has twelve noisy
measurements of one latent content vector, and the cross-reader mean is a strictly better content estimate than any
single row. An EMA prototype bank holds that mean and hands it back as a teacher — plus a second term that makes each
reading pick its *own* prototype out of every prototype the bank knows. That second term is the evaluation, moved
into the loss and scored EEG-to-EEG, so the modality gap cannot be what separates the answer from the distractors.
The bank is written only while training and never consulted at inference: a held-out subject neither enters it nor
reads from it.

**Length-matched gallery contrast.** A batch of sixteen asks the model to beat fifteen distractors; the evaluation
asks it to beat 699, and the hardest of those — same length, same passage, same register — are almost never in a
batch. The frozen text matrix is already resident, so widening the denominator costs one matrix product. Restricting
it to texts of the same word count closes the worse gap: **counting words becomes worth nothing**, so whatever the
loss learns instead is not that. It is the training-time counterpart of length-stratified evaluation.

**Length projection.** Length-stratified evaluation asks whether a hit would survive if length were held constant.
This asks the stronger question — remove the train-fitted length subspace from the exported embeddings, then measure
what is left. It changes no gradient, only what the evaluation reads, and it reports `length_leakage_before` and
`length_leakage_after` so the projection has to show it removed length rather than merely shrinking the vectors.

## 7 · Train the encoder

Every cell here is **resumable**. Re-run it verbatim after an interruption: finished runs are skipped, an
interrupted one continues from its last checkpoint, and each stage mirrors to Drive as it completes.

### 7a · Pick an arm

The dropdown reads `experiments/` live, so it always offers what is actually on disk.

In [ ]:
ENCODER_ARMS: dict[str, str] = {
    'exp16 • the new encoder (all four mechanisms)': 'experiments/flagship/zte_encoder_v3.yaml',
    'exp14 • lexical alignment only': 'experiments/flagship/zte_lexical_raw.yaml',
    'exp12 • best measured on real ZuCo': 'experiments/flagship/zte_raw_aligned.yaml',
    'exp8 • CLIP against E5, raw frontend': 'experiments/flagship/clip_e5_raw.yaml',
}
HOLDOUTS: list[str] = ['ZAB', 'ZDM', 'ZDN', 'ZGW', 'ZJM', 'ZJN', 'ZJS', 'ZKB', 'ZKH', 'ZKW', 'ZMG', 'ZPH']

CONFIG: str = 'experiments/flagship/zte_encoder_v3.yaml'
HOLDOUT: str = 'ZAB'
SEED: int = 42


def _picker() -> None:
    """Offers the arm / held-out subject / seed as widgets, falling back to the assignments above off Colab."""
    try:
        import ipywidgets as widgets
        from IPython.display import display
    except ImportError:
        print(f'ipywidgets unavailable — edit CONFIG / HOLDOUT / SEED above.\n  {CONFIG} · {HOLDOUT} · s{SEED}')
        return

    arm = widgets.Dropdown(options=list(ENCODER_ARMS), value=next(iter(ENCODER_ARMS)), description='arm:')
    holdout = widgets.Dropdown(options=HOLDOUTS, value=HOLDOUT, description='hold out:')
    seed = widgets.IntText(value=SEED, description='seed:')
    out = widgets.Output()

    def _sync(_: object = None) -> None:
        globals().update(CONFIG=ENCODER_ARMS[arm.value], HOLDOUT=holdout.value, SEED=int(seed.value))
        with out:
            out.clear_output()
            print(f'{globals()["CONFIG"]}\n  hold out {globals()["HOLDOUT"]} · seed {globals()["SEED"]}')

    for control in (arm, holdout, seed):
        control.observe(_sync, names='value')
    _sync()
    display(widgets.VBox([widgets.HBox([arm, holdout, seed]), out]))


_picker()

### 7b · Run it

One config × one held-out subject. On a raw-conformer arm with 40 epochs this is a few hours; it checkpoints every
epoch and mirrors to Drive, so an interruption costs at most one epoch.

`--loso-holdout` forces the honest split, `--seed` pins the run, and `--data-cache` points at the staged bundle so
nothing is re-prepared.

In [ ]:
RUN_NAME: str = f'{pathlib.Path(CONFIG).stem}_lo{HOLDOUT}_s{SEED}'
print('->', RUN_NAME)

!uv run zte-run \
  --config "{CONFIG}" \
  --root "{DATA_DIR}" \
  --name "{RUN_NAME}" \
  --out-root "{OUT_ROOT}" \
  --loso-holdout "{HOLDOUT}" \
  --seed {SEED} \
  --data-cache "{PREPARED_LOCAL}" \
  --drive-backup "{DRIVE_BACKUP}" \
  --spatial exact \
  --resume

### 7c · The four ablations — one lever each

Every arm below is byte-identical to `zte_encoder_v3.yaml` except for the single named lever, so a difference in
held-out rank percentile is attributable to that lever and nothing else. This is what makes the exp16 mechanisms
falsifiable rather than merely present.

| arm | lever | the question it answers |
| --- | --- | --- |
| `exp16_residual_off` | `model.residual_coding` | Does de-trending against context raise content and lower the subject probe? |
| `exp16_consensus_off` | the three `consensus_*` weights | Is the cross-reader mean a better target than the text alone? |
| `exp16_gallery_off` | `objective.gallery_weight` | Does a 700-wide denominator beat a 15-wide one at all? |
| `exp16_gallery_band_off` | `objective.gallery_length_band` | And what does length matching add on top of that? |
| `exp16_length_projection_off` | `objective.length_projection` | How much of the headline was word count? |

`exp16_length_projection_off` changes no gradient — only what the evaluation is measured on — so read it as a
measurement of the confound, not as a training result.

In [ ]:
ABLATIONS: list[str] = [
    'experiments/ablation/exp16_residual_off.yaml',
    'experiments/ablation/exp16_consensus_off.yaml',
    'experiments/ablation/exp16_gallery_off.yaml',
    'experiments/ablation/exp16_gallery_band_off.yaml',
    'experiments/ablation/exp16_length_projection_off.yaml',
]

for config in ABLATIONS:
    name = f'{pathlib.Path(config).stem}_lo{HOLDOUT}_s{SEED}'
    print(f'\n=== {name} ' + '=' * 40)
    !uv run zte-run \
      --config "{config}" --root "{DATA_DIR}" --name "{name}" --out-root "{OUT_ROOT}" \
      --loso-holdout "{HOLDOUT}" --seed {SEED} --data-cache "{PREPARED_LOCAL}" \
      --drive-backup "{DRIVE_BACKUP}" --spatial exact --resume

mirror_to_drive()

### 7d · Multi-seed — the error bars a reviewer will ask for

A single seed is not a result on this corpus: the 2026-07-25 sweep moved between 2 and 9 hits in 700 across arms
whose *only* difference was noise, and an earlier re-run of one identical configuration gave 4 hits and then 2.

Three to five fixed seeds, reported as **mean ± sd**. The analysis section aggregates them automatically.

In [ ]:
SEEDS: tuple[int, ...] = (42, 43, 44)

for seed in SEEDS:
    name = f'{pathlib.Path(CONFIG).stem}_lo{HOLDOUT}_s{seed}'
    print(f'\n=== {name} ' + '=' * 40)
    !uv run zte-run \
      --config "{CONFIG}" --root "{DATA_DIR}" --name "{name}" --out-root "{OUT_ROOT}" \
      --loso-holdout "{HOLDOUT}" --seed {seed} --data-cache "{PREPARED_LOCAL}" \
      --drive-backup "{DRIVE_BACKUP}" --spatial exact --resume

mirror_to_drive()

### 7e · The full LOSO sweep — twelve strangers

One config against every subject in turn. This is the exhaustive *does it reach a brain it has never seen* trend and
the only number allowed to be called generalisation. **Multi-hour**, and resumable.

Read the result with `zte-loso-summary`, never with the per-fold pooled Top-1 in `INDEX.md`.

In [ ]:
!SPATIAL=exact DATA_CACHE="{PREPARED_LOCAL}" FULL_CFG="{CONFIG}" \
 DRIVE_BACKUP="{DRIVE_DIR}/loso" OUT_ROOT="{OUT_ROOT}/loso" \
 bash scripts/run_loso.sh "{DATA_DIR}"

!uv run zte-loso-summary --experiments "{OUT_ROOT}/loso" --out "{DRIVE_ANALYSIS}/LOSO.md"
mirror_to_drive(f'{OUT_ROOT}/loso', 'loso')

## 8 · The decoder — text out, on a short leash

The decoder reads a frozen LM through a small trainable bridge. Two mechanisms make it auditable rather than merely
fluent:

**The semantic rate ladder.** The conditioning vector passes through residual codebooks seeded by k-means on the
frozen *text* cloud, so the channel carries at most $\text{stages} \times \log_2(\text{codes})$ bits **by
construction**, and `bit_budget` reports how many actually arrived against the 9.45 needed. The bit budget stops
being an argument and becomes an instrument. Stage 0 is reserved for word count and the rest are penalised for
correlating with it, so `residual_mutual_information_bits` is the part the brain supplied.

**Word-synchronous lexical evidence.** A monotonic pointer walks the reading's words as the LM decodes — eye tracking
gives that alignment for free — nudging the LM's final hidden state, which through a linear frozen head *is* a
rank-limited logit bias. The pointer schedule is **content-free by construction**, which is what makes the
`length_only` control fair: it keeps the schedule and zeroes the content.

### The seven controls

Free-running generation is scored against every one of them, and the verdict fails if any is missing.

| control | what it removes | what it proves if the decode still wins |
| --- | --- | --- |
| `mean_prefix` | the reading, keeping the average | the answer is in *this* reading, not the cohort mean |
| `null_prefix` | the prefix entirely | the LM is not just being an LM |
| `phase` | the EEG's phase, keeping its spectrum | the signal is temporal structure, not band power |
| `noise` | the EEG, keeping the shape | Gaussian noise scoring the same means hallucinated priors |
| `shuffled_z` | the pairing, keeping the distribution | it is *this* brain state, not any brain state |
| `length_only` | the content, keeping the length schedule | it is lexical content, not word count |
| `mismatch` | the correspondence, pairing wrong readings | the alignment is doing the work |

Generation is evaluated **strictly autoregressively** — no ground-truth prefixes, greedy decode, recorded in the
artifact as `teacher_forced: false`. Teacher-forced perplexity is computed and quarantined as `*_DIAGNOSTIC`, never
read by the verdict.

In [ ]:
DECODER_ARMS: dict[str, str] = {
    'exp15 • the v2 decoder (ladder + evidence)': 'experiments/flagship/decode_zte_v2.yaml',
    'v2 • pooled only (reproduces the v1 decoder)': 'experiments/decoder/decode_v2_pooled.yaml',
    'v2 • rate ladder only': 'experiments/decoder/decode_v2_ladder_only.yaml',
    'v2 • evidence only': 'experiments/decoder/decode_v2_evidence_only.yaml',
    'v2 • no reserved length stage': 'experiments/decoder/decode_v2_no_length_stage.yaml',
    'v2 • band-power frontend': 'experiments/decoder/decode_v2_bandpower.yaml',
}
DECODER_CONFIG: str = 'experiments/flagship/decode_zte_v2.yaml'


def _decoder_picker() -> None:
    """Offers the decoder arm as a dropdown, falling back to the assignment above off Colab."""
    try:
        import ipywidgets as widgets
        from IPython.display import display
    except ImportError:
        print(f'ipywidgets unavailable — edit DECODER_CONFIG above.\n  {DECODER_CONFIG}')
        return

    arm = widgets.Dropdown(options=list(DECODER_ARMS), value=next(iter(DECODER_ARMS)), description='decoder:')
    out = widgets.Output()

    def _sync(_: object = None) -> None:
        globals()['DECODER_CONFIG'] = DECODER_ARMS[arm.value]
        with out:
            out.clear_output()
            print(globals()['DECODER_CONFIG'])

    arm.observe(_sync, names='value')
    _sync()
    display(widgets.VBox([arm, out]))


_decoder_picker()

### 8a · Train the bridge over the frozen encoder

The decoder is trained over a **frozen** encoder checkpoint, so the number it produces is attributable to the bridge
and not to a quietly-retrained encoder. Point `--encoder-from` at the arm you trained in §7.

In [ ]:
# Drive first: on a fresh runtime the encoder you trained last session is on Drive and not on this disk.
ENCODER_CKPT: str = resolve_ckpt(RUN_NAME)
DECODER_NAME: str = f'{pathlib.Path(DECODER_CONFIG).stem}_lo{HOLDOUT}_s{SEED}'
print('decoder ->', DECODER_NAME)

!uv run zte-run \
  --config "{DECODER_CONFIG}" --root "{DATA_DIR}" --name "{DECODER_NAME}" --out-root "{OUT_ROOT}" \
  --encoder-ckpt "{ENCODER_CKPT}" --seed {SEED} --data-cache "{PREPARED_LOCAL}" \
  --drive-backup "{DRIVE_BACKUP}" --resume

### 8b · Decode with every control, over several seeds

`zte-decode` runs the headline decode and all seven controls through **one** code path, so the comparison is paired
rather than approximate. `--seeds` repeats the whole thing and reports the spread of the worst-control delta, which
is the quantity the verdict actually gates on.

In [ ]:
!uv run zte-decode \
  --ckpt "{resolve_ckpt(DECODER_NAME)}" \
  --root "{DATA_DIR}" --split test \
  --controls mean_prefix,null_prefix,phase,noise,shuffled_z,length_only,mismatch \
  --seeds 42,43,44 \
  --within-task \
  --out "{DRIVE_ANALYSIS}/decode_{DECODER_NAME}"

### 8c · The length audit — read this before any decoder number

`zte-rebaseline` measures how much of a checkpoint's retrieval a length-only oracle reproduces, with no retraining.
It gates nothing; it tells you which column of the report to trust.

On the current best decoder, length-stratified rescoring rank percentile was **0.4349** — below the 0.5 chance line.
That is the whole reason the reserved length stage and the `length_only` control exist.

In [ ]:
!uv run zte-rebaseline \
  --ckpt "{resolve_ckpt(DECODER_NAME)}" \
  --root "{DATA_DIR}" \
  --out "{DRIVE_ANALYSIS}/rebaseline_{DECODER_NAME}.json"

### 8d · Read the decodes — target beside hypothesis beside every control

The table below is the plain answer to *how is it doing*: for each held-out reading, the sentence the person read,
the sentence the decoder wrote from their EEG, and the same decode from each brain-independent condition.

Read it in this order, and only in this order:

1. **The controls first.** A frozen LM reaches ROUGE-1 in the 0.10–0.18 range against *any* English reference from
   function words alone, so an absolute score is not evidence of anything. `null_prefix` is the floor the language
   model gets for free.
2. **Then the paired delta.** Hypothesis minus control, on the same reading — that is the only quantity that carries
   information about the brain.
3. **Then remember the arithmetic.** The encoder supplies ~1.5 bits and a 19.6-word sentence needs ~190. An honest
   null here is the expected result, and reporting it plainly is the finding.

`length_only` is the control that matters most for this project. It keeps the pointer schedule — so it still gets the
word count, ZuCo's free 5.14 bits — and destroys only *what each word was*. A hypothesis that beats it beat it on
lexical content and on nothing else.

In [ ]:
from dataclasses import replace

import numpy as np
import pandas as pd
from IPython.display import IFrame

from zte.cli.decode import split_indices
from zte.config import ZTEConfig
from zte.data.dataset import ZuCoDataset
from zte.evaluation.generation import content_word_f1, sentence_wer
from zte.inference.decode import ZTEDecoder, paired_shuffle
from zte.training.checkpoint import CheckpointManager

DECODE_CKPT: str = resolve_ckpt(DECODER_NAME)
N_READINGS: int = 12

# The checkpoint's own dataset config supplies every processing option, so a raw-frontend run finds raw tensors and
# a band-power run finds the exact feature width it was trained at. Only the root is repointed at this session's.
payload = CheckpointManager.load(DECODE_CKPT, map_location='cpu')
decode_config = ZTEConfig.from_dict(payload['config'])
dataset = ZuCoDataset(replace(decode_config.dataset, root=DATA_DIR)).build(show_progress=False)

zte_decoder = ZTEDecoder.from_checkpoint(DECODE_CKPT, dataset)
held_out = zte_decoder.conditioning(dataset, split_indices(dataset, decode_config, 'test'))
print(f'{len(held_out)} held-out readings')

rows = np.unique(np.linspace(0, len(held_out) - 1, min(N_READINGS, len(held_out))).astype(int))
subset = held_out.take(rows)
subset.meta = held_out.meta.iloc[rows].reset_index(drop=True)
references = [str(t) for t in subset.meta['text']]

decoded: dict[str, list[str]] = {
    'hypothesis (EEG)': zte_decoder.generate(subset),
    'null_prefix': zte_decoder.generate_from_prefix(zte_decoder.null_prefix(len(subset))),
    'length_only': zte_decoder.generate(subset, evidence_content=False),
    'mismatch': zte_decoder.generate(subset.take(paired_shuffle(len(subset), seed=0))),
}

scored = pd.DataFrame(
    [
        {
            'condition': name,
            'reading': i,
            'subject': subset.meta.iloc[i]['subject'],
            'n_words': int(subset.meta.iloc[i]['n_words']),
            'text': text[i],
            'content_f1': float(content_word_f1([text[i]], [references[i]])[0]),
            'wer': float(sentence_wer(text[i], references[i])),
        }
        for name, text in decoded.items()
        for i in range(len(subset))
    ]
)
display(scored.pivot_table(index='condition', values=['content_f1', 'wer'], aggfunc='mean').round(4))

# The paired delta is the readable quantity: hypothesis minus control, on the same reading.
wide = scored.pivot(index='reading', columns='condition', values='content_f1')
deltas = wide.drop(columns=['hypothesis (EEG)']).rsub(wide['hypothesis (EEG)'], axis=0)
display(deltas.describe().loc[['mean', 'std', 'min', 'max']].round(4))

In [ ]:
# One reading at a time: the sentence read, the sentence written, and each control's attempt at the same row.
for i in range(min(4, len(subset))):
    meta = subset.meta.iloc[i]
    print('=' * 104)
    print(f'{meta["subject"]} · {meta["task"]} · {int(meta["n_words"])} words')
    print(f'  TARGET       : {references[i]}')
    for name, text in decoded.items():
        f1 = float(content_word_f1([text[i]], [references[i]])[0])
        print(f'  {name:<13}: {text[i][:104]!r}   content_f1={f1:.4f}')

### 8e · The decode studio — watch it happen

`zte-studio` decodes a handful of held-out readings **with a full per-step trace** and writes one self-contained
interactive page. It is the same `generate_from_prefix` call the evaluation makes — the trace is a sink the loop
writes to and nothing else, so the page cannot show a decode the evaluation did not make.

What is on it, and the real quantity behind each part:

| panel | what it is actually reading |
| --- | --- |
| **Scalp field** (2D cap · draggable 3D head) | per-word band power interpolated across the montage, at the word the pointer is on |
| **Target sentence** | the pointer's Gaussian window over the reading's words, at the current decoding step |
| **Decoded text** | tokens revealed as they were emitted, shaded by probability; click one to jump to that step |
| **Alternatives** | the top-8 next-token distribution at that step — what it nearly said |
| **Evidence KL** | the same hidden state with and without the word-synchronous nudge: how hard the brain pushed on *this* token |
| **Pointer walk** | the full (step × word) attention matrix |
| **Rate-ladder codes** | which codebook entry each stage selected for this reading |

Space plays and pauses, the arrow keys step, and the scrub bar seeks. The band buttons switch which rhythm the scalp
map shows — theta and gamma are the lexical-semantic pair, alpha and beta the attentional one.

**The scalp colour scale is relative within one reading**, so two readings whose maps look alike are not therefore
alike in microvolts. The page says so on itself.

> **It is an inspection tool, not an audit.** A handful of readings chosen to look at is an anecdote; the verdict
> needs the paired delta over every held-out reading, its bootstrap interval and the permutation null. The page
> carries that warning in its own banner so a screenshot cannot be mistaken for a result.

In [ ]:
STUDIO: str = f'{DRIVE_ANALYSIS}/STUDIO_{DECODER_NAME}.html'

!uv run zte-studio \
  --ckpt "{DECODE_CKPT}" \
  --root "{DATA_DIR}" \
  --split test \
  --rows 8 \
  --controls null_prefix,length_only,mismatch \
  --montage res/montage_gsn105.csv \
  --out "{STUDIO}"

# Copy to the VM disk before embedding: an iframe reading straight off the Drive mount is slow enough to look broken.
local_studio = pathlib.Path('res/analysis/STUDIO.html')
local_studio.parent.mkdir(parents=True, exist_ok=True)
shutil.copyfile(STUDIO, local_studio)
print(f'{local_studio}  ({local_studio.stat().st_size / 1e6:.1f} MB)  ·  also on Drive at {STUDIO}')
display(IFrame(src=str(local_studio), width='100%', height=900))

## 9 · The whole study in one command

`scripts/run_zte_study.sh` runs every stage in order and is safe to re-run after any interruption — each stage skips
work already on disk and mirrors to Drive as it finishes.

| stage | what it does |
| --- | --- |
| `audit` | the model-free confound report |
| `encoder` | the flagship encoder arms |
| `loso` | the full twelve-fold sweep |
| `decoder` | the decoder arms and their controls |
| `ablation` | one-lever studies |
| `rebaseline` | the length-oracle audit |
| `analysis` | the dashboard and its tidy tables |

Set `STAGES` to a subset to run only part of it. The dashboard lands in `$OUT_ROOT/analysis` and is mirrored to
`$DRIVE_BACKUP/analysis`, so it survives the VM either way. **This is the multi-hour cell** — start it, and come
back.

In [ ]:
!SEEDS="42 43 44" \
 STAGES="audit encoder loso decoder ablation rebaseline analysis" \
 SPATIAL=exact \
 DATA_CACHE="{PREPARED_LOCAL}" \
 OUT_ROOT="{OUT_ROOT}" \
 DRIVE_BACKUP="{DRIVE_BACKUP}" \
 bash scripts/run_zte_study.sh "{DATA_DIR}"

mirror_to_drive()

## 10 · Analysis — everything you have ever run, visually

`zte-analyze` walks any number of run trees, collects every artifact into tidy frames, and writes one **self-contained
offline HTML page** plus the CSVs behind it. Plotly is inlined, so the page opens from a Drive mirror on a machine
with no network.

It reads across **every session folder on Drive**, not just today's, so the picture is cumulative.

In [ ]:
ANALYSIS_ROOTS: list[str] = [*every_session(), LOCAL_RUNS]
print('reading:')
for root in ANALYSIS_ROOTS:
    print('  ', root)

roots = ' '.join(f'"{r}"' for r in ANALYSIS_ROOTS)
!uv run zte-analyze --experiments {roots} --out "{DRIVE_ANALYSIS}" --montage res/montage_gsn105.csv

### 10a · The tables that carry the argument

Four frames, each answering one question. Every cell is **mean ± sd across seeds**, and `n_seeds` travels with it so
a single-run row cannot be mistaken for a stable one.

In [ ]:
import pandas as pd

from zte.evaluation.analysis import (
    collect_study,
    feature_ablation_table,
    loso_table,
    multi_seed_table,
    within_task_table,
)

pd.set_option('display.max_columns', 60, 'display.width', 200, 'display.float_format', '{:.4f}'.format)
study = collect_study(ANALYSIS_ROOTS)
print(f'{len(study.runs)} run(s) · {len(study.folds)} fold(s) · {len(study.probes)} probe row(s)')

KEEP: list[str] = [
    'arm',
    'n_seeds',
    'held_out_rank_percentile_mean',
    'held_out_rank_percentile_sd',
    'held_out_top1_mean',
    'stratified_rank_percentile_mean',
    'effective_rank_ratio_mean',
]
table = multi_seed_table(study)
display(table[[c for c in KEEP if c in table]].sort_values('held_out_rank_percentile_mean', ascending=False))

In [ ]:
for name, frame in (
    ('Per-lever ablation', feature_ablation_table(study)),
    ('LOSO, per held-out subject', loso_table(study)),
    ('Within-task pools (SR / NR)', within_task_table(study)),
):
    print(f'\n### {name}')
    display(frame if not frame.empty else 'nothing collected for this view yet')

### 10b · The panels, inline and interactive

The full page has every chart; these are the ones worth having in the notebook, where you can hover and filter them
next to the code that produced them. Each returns `None` rather than a misleading empty axis when the data behind it
was not collected.

- **Pick a headline** — every metric behind one dropdown.
- **Did the mechanism engage?** — per-epoch training curves. A consensus term that never fired or a gallery accuracy
  pinned at chance is visible *only* here; the final metrics cannot tell "did nothing" from "was never switched on".
- **Who vs what** — the subject probe against the content probe, sized by effective rank, because a probe that falls
  on both axes has collapsed rather than become invariant.
- **Length leakage** — before and after the projection.
- **Bit budget** — the 9.45 bits, and who supplies them.
- **Electrode map** — the scalp geometry the encoder actually reads.

In [ ]:
from zte.evaluation.analysis import figures as F

PANELS = [
    ('Pick a headline', lambda: F.metric_explorer(study)),
    ('Did the mechanism engage?', lambda: F.mechanism_curves(study)),
    ('Who vs what', lambda: F.identity_vs_content(study)),
    ('Length leakage, before and after', lambda: F.length_leakage_bars(study)),
    ('The encoder against a length oracle', lambda: F.length_confound_scatter(study)),
    ('Bit budget', lambda: F.bit_budget_pie(study)),
    ('Every run as one observation', lambda: F.seed_histogram(study)),
    ('Which levers were on', lambda: F.mechanism_matrix(study)),
    ('LOSO — which brains a recipe reaches', lambda: F.loso_heatmap(study)),
    ('Fold-to-fold spread', lambda: F.fold_spread(study)),
    ('What each representation carries', lambda: F.probe_heatmap(study)),
    ('Who versus what, by variance', lambda: F.variance_budget_pie(study)),
    ('The decode against every control', lambda: F.control_ladder(study)),
    ('Where the decode landed, sentence by sentence', lambda: F.text_overlap_heatmap(study)),
    ('The words emitted against the words asked for', lambda: F.word_frequency_bars(study)),
    ('Electrode geometry', lambda: F.scalp_3d(montage_csv='res/montage_gsn105.csv')),
]

for title, build in PANELS:
    figure = build()
    if figure is None:
        print(f'— {title}: no data collected for this panel yet')
        continue
    figure.show()

### 10c · Drill into one run

The dropdown lists every run the analysis found. Selecting one prints its honest headline block and shows its saved
figures, so a suspicious number can be chased to the run that produced it without leaving the notebook.

In [ ]:
def _run_explorer() -> None:
    """A dropdown over every collected run that prints its headline block and displays its figures."""
    directories = run_dirs(*ANALYSIS_ROOTS)
    if not directories:
        print('no runs found yet')
        return

    try:
        import ipywidgets as widgets
        from IPython.display import Image, display
    except ImportError:
        print('ipywidgets unavailable — runs found:', ', '.join(d.name for d in directories))
        return

    picker = widgets.Dropdown(options=[d.name for d in directories], description='run:')
    by_name = {d.name: d for d in directories}
    out = widgets.Output()

    def _show(_: object = None) -> None:
        directory = by_name[picker.value]
        with out:
            out.clear_output()
            metrics_path = directory / 'evaluation' / 'metrics.json'
            if not metrics_path.is_file():
                print('not evaluated yet:', directory)
                return

            metrics = json.loads(metrics_path.read_text())
            board = metrics.get('scoreboard', {})
            held = board.get('held_out_retrieval') or {}
            print(f'{directory.name}   held out: {board.get("holdout_subject")}')
            print(f'  rank percentile : {held.get("rank_percentile")}')
            print(f'  Top-1           : {held.get("top1")}  (chance {held.get("chance_top1")})')
            print(f'  postprocess fit : {metrics.get("postprocess_fit")}')
            print(f'  length projection: {metrics.get("length_projection")}')
            print(f'  verdict         : {json.dumps(board.get("verdict", {}), indent=2)[:600]}')
            for figure in sorted((directory / 'evaluation' / 'figures').glob('*.png'))[:8]:
                display(Image(filename=str(figure), width=620))

    picker.observe(_show, names='value')
    _show()
    display(widgets.VBox([picker, out]))


_run_explorer()

### 10d · The full page

A few megabytes with Plotly inlined, so the frame below can be sluggish — the file itself is on Drive and opens
fastest in its own tab. It is also the artifact to share: no server, no network, no dependencies.

In [ ]:
from IPython.display import HTML, IFrame

page = pathlib.Path(DRIVE_ANALYSIS) / 'ANALYSIS.html'
print(page, '·', f'{page.stat().st_size / 1e6:.1f} MB' if page.is_file() else 'not built yet')
print((pathlib.Path(DRIVE_ANALYSIS) / 'ANALYSIS.md').read_text()[:2000])

local_copy = pathlib.Path('res/analysis/ANALYSIS.html')
if page.is_file():
    local_copy.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(page, local_copy)
    display(IFrame(src=str(local_copy), width='100%', height=760))
else:
    display(HTML('<p>Run §10 first.</p>'))

## 11 · Persist, resume, continue offline

Three levels, and they answer different questions.

- **Mirror** — a browsable copy of the runs on Drive. Already automatic after each stage; call it again any time.
- **Archive** — a provenance-stamped zip of the best checkpoints, so a result can be reproduced later.
- **Snapshot** — the entire working state, *including the prepared data cache*, in one file. Download it and keep
  working on your own machine with no GPU time: `zte-pack unpack <zip> --dest res`.

If the runtime resets, `restore_from_drive()` pulls the session back to the VM and every training cell picks up from
its last checkpoint.

In [ ]:
def archive_to_drive(note: str | None = None) -> None:
    """Provenance-stamped zip of the best checkpoints, skipping synthetic smoke runs."""
    stamp = datetime.datetime.now().strftime('%H%M%S')
    out = f'{DRIVE_DIR}/archives/zte_{RUN_DATE}_{stamp}.zip'
    command = ['uv', 'run', 'zte-pack', 'zip', '--all', '--best-only', '--skip-synthetic', '--out', out]
    subprocess.run([*command, *(['--note', note] if note else [])], check=False)
    print('archive ->', out)


def snapshot_to_drive(note: str | None = None) -> str:
    """Everything -- runs, cache, benchmark, explorer -- in one zip, so a local session never re-prepares data."""
    stamp = datetime.datetime.now().strftime('%H%M%S')
    out = f'{DRIVE_DIR}/archives/zte_snapshot_{RUN_DATE}_{stamp}.zip'
    command = ['uv', 'run', 'zte-pack', 'snapshot', '--skip-synthetic', '--out', out]
    subprocess.run([*command, *(['--note', note] if note else [])], check=False)
    print('snapshot ->', out)
    return out


mirror_to_drive()
archive_to_drive(note=f'{RUN_DATE} encoder v3 + decoder v2')

## 12 · Running it on your own machine

Nothing here is Colab-specific below the Drive cells. On a workstation with a real GPU the same eighteen entry points
do the same work:

```sh
uv sync                                              # 'all' and 'dev' are default groups
uv run zte-prepare  --root <data> --configs          # build the feature bundles once

uv run zte-run      --config experiments/flagship/zte_encoder_v3.yaml \
                    --root <data> --loso-holdout ZAB --seed 42 --resume
uv run zte-audit    --root <data>                    # the model-free confound report
uv run zte-decode   --ckpt <ckpt> --root <data> --split test --seeds 42,43,44
uv run zte-rebaseline --ckpt <ckpt> --root <data>    # how much of a number is sentence length
uv run zte-loso-summary --experiments res/experiments/loso
uv run zte-analyze  --experiments res/experiments --out res/analysis
uv run zte-ablate   generate --config <cfg> --knob objective.gallery_length_band --values 0,1,2,4

SEEDS='42 43 44' bash scripts/run_zte_study.sh <data>   # the whole study, resumable
```

**Device support.** CPU, CUDA, MPS and XLA are all live through `zte.device.resolve_device`. MPS has a ~30 GiB
ceiling and misses operators the CUDA path has — `raw_conformer` at batch 64 will not fit — so a local Apple machine
is for inference, analysis and the synthetic smoke path, not for training the raw arms.

**Before reporting anything**, run the gates the repository is held to:

```sh
uv run ruff format . && uv run ruff check . && uv run mypy src tests && uv run pytest
```

---

## 13 · Housekeeping

The VM disk fills up faster than you expect, mostly with checkpoints you have already mirrored.

In [ ]:
!uv run zte-pack list


def remove_locally(*names: str) -> None:
    """Delete local run directories or res/ subpaths to free disk. Never touches Drive."""
    for name in names:
        path = pathlib.Path(name)
        if not path.exists():
            path = pathlib.Path(LOCAL_RUNS) / name
        if path.exists():
            shutil.rmtree(path)
            print('removed', path)
        else:
            print('not found:', name)


# remove_locally('res/experiments/smoke_run', 'res/cache')
show_resources()